In [2]:
import pandas as pd, numpy as np

expr_rich = pd.read_csv("log2_tpm_rich_zscore.csv", index_col=0)
gene_cols = [c for c in expr_rich.columns]  # HGNC symbols, dots replace hyphens (e.g. HLA.F)

def clean_symbol(s):
    return s.replace(".", "-")

expr_genes = set(clean_symbol(g) for g in gene_cols)
print(len(expr_genes))  # ~14890

14890


In [3]:
pip install mygene

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import mygene

# STRING
string_info = pd.read_csv("9606.protein.info.v12.0.txt.gz", sep="\t", compression="gzip")
STRING_genes = set(string_info["preferred_name"].dropna())
print("STRING genes:", len(STRING_genes))

# Reactome
print("Loading reactome...")
reactome = pd.read_csv(
    r"D:\Final Year 2\Data\download-reactome-org-97-NCBI2Reactome-txt.txt",
    sep="\t", header=None,
    names=["gene_id","reactome_id","url","pathway","evidence","species"],
    low_memory=False
)
reactome_hs = reactome[reactome["species"] == "Homo sapiens"].copy()
reactome_hs["gene_id"] = pd.to_numeric(reactome_hs["gene_id"], errors="coerce")
reactome_hs = reactome_hs.dropna(subset=["gene_id"])
reactome_hs["gene_id"] = reactome_hs["gene_id"].astype(int)
ncbi_ids = set(reactome_hs["gene_id"].unique())
print("Unique NCBI IDs:", len(ncbi_ids))

print("Querying mygene API... wait 2-5 mins")
mg = mygene.MyGeneInfo()
result = mg.getgenes(list(ncbi_ids), fields="symbol", species="human", as_dataframe=True)
result = result.dropna(subset=["symbol"])
id_to_symbol = dict(zip(result.index.astype(int), result["symbol"]))
REACTOME_genes = set(id_to_symbol.values())
print("Reactome genes (symbols):", len(REACTOME_genes))

STRING genes: 19699
Loading reactome...
Unique NCBI IDs: 11766
Querying mygene API... wait 2-5 mins
Reactome genes (symbols): 11363


In [11]:
import numpy as np

# expr_genes should already be defined from your first cell
# if not, rebuild it:
expr_rich = pd.read_csv("log2_tpm_rich_zscore.csv", index_col=0)
gene_cols = list(expr_rich.columns)
expr_genes = set(g.replace(".", "-") for g in gene_cols)
print("Expr genes:", len(expr_genes))

# Intersection
shared_genes = sorted(expr_genes & STRING_genes & REACTOME_genes)
gene_idx = {g: i for i, g in enumerate(shared_genes)}
print("Shared genes (intersection):", len(shared_genes))

np.save("shared_genes.npy", np.array(shared_genes))
print("Saved shared_genes.npy")

Expr genes: 14890
Shared genes (intersection): 9253
Saved shared_genes.npy


In [12]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler

print("Building co-expression graph...")

# Subset expression matrix to shared genes
col_map = {c: c.replace(".", "-") for c in expr_rich.columns}
expr_rich_renamed = expr_rich.rename(columns=col_map)
expr_shared = expr_rich_renamed[shared_genes]  # shape: (127, 9253)
print("Expression matrix shape:", expr_shared.shape)

# Compute Spearman correlation across cell lines (gene x gene)
# This gives a population-level co-expression topology
print("Computing Spearman correlation (takes 2-4 mins)...")
corr_matrix, _ = spearmanr(expr_shared.values)  # (9253, 9253)
print("Correlation matrix shape:", corr_matrix.shape)

# Threshold at |r| >= 0.6 → edges
COEXPR_THRESHOLD = 0.6
adj = (np.abs(corr_matrix) >= COEXPR_THRESHOLD).astype(np.uint8)
np.fill_diagonal(adj, 0)  # remove self-loops

edge_count = adj.sum() // 2
print(f"Co-expression edges (|r|>=0.6): {edge_count}")

# Save as edge index (COO format for PyG)
src, dst = np.where(adj)
coexpr_edge_index = np.stack([src, dst], axis=0)  # (2, E)
np.save("coexpr_edge_index.npy", coexpr_edge_index)
print("Saved coexpr_edge_index.npy")

Building co-expression graph...
Expression matrix shape: (127, 9253)
Computing Spearman correlation (takes 2-4 mins)...


c:\Users\ANJANA\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\ANJANA\AppData\Local\Programs\Python\Python311\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


Correlation matrix shape: (9253, 9253)
Co-expression edges (|r|>=0.6): 15221
Saved coexpr_edge_index.npy


In [ ]:
print("Building STRING PPI graph...")

# Load STRING links
string_links = pd.read_csv(
    "9606.protein.links.v12.0.txt.gz",
    sep=" ", compression="gzip"
)
print("STRING links loaded:", len(string_links))

# Map ENSP → gene symbol using string_info
ensp_to_symbol = dict(zip(string_info["#string_protein_id"], string_info["preferred_name"]))

string_links["gene1"] = string_links["protein1"].map(ensp_to_symbol)
string_links["gene2"] = string_links["protein2"].map(ensp_to_symbol)
``
# Filter: both genes in shared set, score >= 700
string_filtered = string_links[
    (string_links["combined_score"] >= 700) &
    (string_links["gene1"].isin(gene_idx)) &
    (string_links["gene2"].isin(gene_idx))
].copy()
print("Filtered STRING edges:", len(string_filtered))

src = string_filtered["gene1"].map(gene_idx).values
dst = string_filtered["gene2"].map(gene_idx).values
ppi_edge_index = np.stack([src, dst], axis=0)  # (2, E)
ppi_edge_weights = string_filtered["combined_score"].values / 1000.0  # normalize to [0,1]

np.save("ppi_edge_index.npy", ppi_edge_index)
np.save("ppi_edge_weights.npy", ppi_edge_weights)
print("Saved ppi_edge_index.npy and ppi_edge_weights.npy")

Building STRING PPI graph...
STRING links loaded: 13715404
Filtered STRING edges: 252698
Saved ppi_edge_index.npy and ppi_edge_weights.npy


In [14]:
print("Building Reactome pathway graph...")

# Group genes by pathway
pathway_to_genes = reactome_hs.copy()
pathway_to_genes["symbol"] = pathway_to_genes["gene_id"].map(id_to_symbol)
pathway_to_genes = pathway_to_genes.dropna(subset=["symbol"])
pathway_to_genes = pathway_to_genes[pathway_to_genes["symbol"].isin(gene_idx)]

# For each pathway, connect all gene pairs (clique)
from itertools import combinations

edges = set()
grouped = pathway_to_genes.groupby("reactome_id")["symbol"].apply(list)
print("Total pathways:", len(grouped))

for pathway, genes in grouped.items():
    genes_in_shared = [g for g in genes if g in gene_idx]
    if len(genes_in_shared) < 2:
        continue
    for g1, g2 in combinations(genes_in_shared, 2):
        i, j = gene_idx[g1], gene_idx[g2]
        if i != j:
            edges.add((min(i,j), max(i,j)))

print("Reactome pathway edges:", len(edges))
edges = np.array(list(edges))  # (E, 2)

# Make bidirectional
pathway_edge_index = np.stack([
    np.concatenate([edges[:,0], edges[:,1]]),
    np.concatenate([edges[:,1], edges[:,0]])
], axis=0)  # (2, 2E)

np.save("pathway_edge_index.npy", pathway_edge_index)
print("Saved pathway_edge_index.npy")

Building Reactome pathway graph...
Total pathways: 2273
Reactome pathway edges: 629030
Saved pathway_edge_index.npy


In [15]:
import pandas as pd
import numpy as np

def align_expr(csv_path, shared_genes):
    df = pd.read_csv(csv_path, index_col=0)
    df = df.rename(columns={c: c.replace(".", "-") for c in df.columns})
    # fill missing genes with 0
    df = df.reindex(columns=shared_genes, fill_value=0.0)
    return df.values.astype(np.float32)  # (n_samples, 9253)

expr_rich_np   = align_expr("log2_tpm_rich_zscore.csv", shared_genes)
expr_poor_np   = align_expr("log2_tpm_poor_zscorebyrich.csv", shared_genes)
expr_patient_np= align_expr("log2_tpm_patient_zscorebyrich.csv", shared_genes)

print("Rich shape:   ", expr_rich_np.shape)    # (127, 9253)
print("Poor shape:   ", expr_poor_np.shape)    # (13, 9253)
print("Patient shape:", expr_patient_np.shape) # (3, 9253)

Rich shape:    (127, 9253)
Poor shape:    (13, 9253)
Patient shape: (3, 9253)


In [16]:
pip install torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

coexpr_ei   = torch.tensor(np.load("coexpr_edge_index.npy"),  dtype=torch.long).to(device)
ppi_ei      = torch.tensor(np.load("ppi_edge_index.npy"),     dtype=torch.long).to(device)
ppi_ew      = torch.tensor(np.load("ppi_edge_weights.npy"),   dtype=torch.float).to(device)
pathway_ei  = torch.tensor(np.load("pathway_edge_index.npy"), dtype=torch.long).to(device)

print("Coexpr edges: ", coexpr_ei.shape)
print("PPI edges:    ", ppi_ei.shape)
print("Pathway edges:", pathway_ei.shape)

Device: cpu
Coexpr edges:  torch.Size([2, 30442])
PPI edges:     torch.Size([2, 252698])
Pathway edges: torch.Size([2, 1258060])


In [18]:
pip install torch-geometric

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch

class ViewEncoder(nn.Module):
    """Single-view GCN encoder. Input: per-gene scalar expression."""
    def __init__(self, in_dim=1, hidden=128, out_dim=64):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(out_dim)

    def forward(self, x, edge_index, batch, edge_weight=None):
        h = self.conv1(x, edge_index, edge_weight)
        h = self.bn1(h)
        h = F.relu(h)
        h = self.conv2(h, edge_index, edge_weight)
        h = self.bn2(h)
        h = F.relu(h)
        return global_mean_pool(h, batch)  # (B, out_dim)


class CrossViewFusion(nn.Module):
    """Attention-based fusion of 3 view embeddings → single tumor embedding."""
    def __init__(self, view_dim=64, num_views=3, out_dim=128):
        super().__init__()
        # learned attention score per view
        self.attn = nn.Linear(view_dim, 1)
        self.proj = nn.Linear(view_dim, out_dim)
        self.out_dim = out_dim

    def forward(self, v_coexpr, v_ppi, v_pathway):
        # stack: (B, 3, view_dim)
        views = torch.stack([v_coexpr, v_ppi, v_pathway], dim=1)
        # attention weights
        scores = self.attn(views)           # (B, 3, 1)
        weights = F.softmax(scores, dim=1)  # (B, 3, 1)
        # weighted sum
        fused = (weights * views).sum(dim=1)  # (B, view_dim)
        return F.relu(self.proj(fused))        # (B, out_dim)


class TumorEncoder(nn.Module):
    """Wraps all 3 view encoders + fusion."""
    def __init__(self, n_genes=9253, hidden=128, view_dim=64, tumor_dim=128):
        super().__init__()
        self.coexpr_enc  = ViewEncoder(1, hidden, view_dim)
        self.ppi_enc     = ViewEncoder(1, hidden, view_dim)
        self.pathway_enc = ViewEncoder(1, hidden, view_dim)
        self.fusion      = CrossViewFusion(view_dim, 3, tumor_dim)
        self.n_genes     = n_genes

    def forward(self, expr_vec, coexpr_ei, ppi_ei, pathway_ei,
                ppi_ew=None, batch=None):
        """
        expr_vec  : (B*N, 1) — flattened node features for a batch of graphs
        batch     : (B*N,)   — PyG batch vector
        """
        v1 = self.coexpr_enc(expr_vec, coexpr_ei, batch)
        v2 = self.ppi_enc(expr_vec, ppi_ei, batch, ppi_ew)
        v3 = self.pathway_enc(expr_vec, pathway_ei, batch)
        return self.fusion(v1, v2, v3)  # (B, tumor_dim)


# Instantiate
tumor_enc = TumorEncoder(n_genes=9253).to(device)
print(tumor_enc)
total_params = sum(p.numel() for p in tumor_enc.parameters())
print(f"TumorEncoder params: {total_params:,}")

c:\Users\ANJANA\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


TumorEncoder(
  (coexpr_enc): ViewEncoder(
    (conv1): GCNConv(1, 128)
    (conv2): GCNConv(128, 64)
    (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (ppi_enc): ViewEncoder(
    (conv1): GCNConv(1, 128)
    (conv2): GCNConv(128, 64)
    (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (pathway_enc): ViewEncoder(
    (conv1): GCNConv(1, 128)
    (conv2): GCNConv(128, 64)
    (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  )
  (fusion): CrossViewFusion(
    (attn): Linear(in_features=64, out_features=1, bias=True)
    (p

In [20]:
def make_graph_batch(expr_row, coexpr_ei, ppi_ei, pathway_ei, device):
    """
    expr_row: (9253,) numpy array for one cell line
    Returns tensors ready for TumorEncoder.forward()
    """
    n = expr_row.shape[0]
    x = torch.tensor(expr_row, dtype=torch.float32).unsqueeze(1).to(device)  # (N,1)
    batch = torch.zeros(n, dtype=torch.long).to(device)                        # (N,)
    return x, batch

# test with first rich cell line
x_test, batch_test = make_graph_batch(expr_rich_np[0], coexpr_ei, ppi_ei, pathway_ei, device)
with torch.no_grad():
    out = tumor_enc(x_test, coexpr_ei, ppi_ei, pathway_ei, ppi_ew, batch_test)
print("Tumor embedding shape:", out.shape)  # expect (1, 128)
print("Sample values:", out[0, :5])

Tumor embedding shape: torch.Size([1, 128])
Sample values: tensor([0.0000, 0.0000, 0.0000, 0.2972, 0.0000])


In [21]:
pip install rdkit torch-geometric

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [22]:
from rdkit import Chem
from rdkit.Chem import rdmolops
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch

# Atom feature extractor
def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetHybridization()),
        int(atom.GetIsAromatic()),
        atom.GetTotalNumHs(),
    ]

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    # Node features
    x = torch.tensor(
        [atom_features(a) for a in mol.GetAtoms()],
        dtype=torch.float
    )  # (num_atoms, 6)

    # Edge index (bidirectional)
    edges = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges += [[i, j], [j, i]]
    if len(edges) == 0:
        # single atom drug
        edge_index = torch.zeros((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index)

In [23]:
class DrugEncoder(nn.Module):
    def __init__(self, in_dim=6, hidden=128, out_dim=64):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, out_dim)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.bn2   = nn.BatchNorm1d(out_dim)

    def forward(self, x, edge_index, batch):
        h = self.conv1(x, edge_index)
        h = self.bn1(h)
        h = F.relu(h)
        h = self.conv2(h, edge_index)
        h = self.bn2(h)
        h = F.relu(h)
        return global_mean_pool(h, batch)  # (B, out_dim)

drug_enc = DrugEncoder().to(device)
print(drug_enc)
print(f"DrugEncoder params: {sum(p.numel() for p in drug_enc.parameters()):,}")

DrugEncoder(
  (conv1): GCNConv(6, 128)
  (conv2): GCNConv(128, 64)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
)
DrugEncoder params: 9,536


In [24]:
import pandas as pd
import numpy as np

# Load drug file
drugs_df = pd.read_csv("4130_drug_smiles_cid.csv")
print("Columns:", drugs_df.columns.tolist())
print(drugs_df.head(3))

Columns: ['cid', 'ԭcid', 'smiles']
   cid     ԭcid                                             smiles
0    0  5287792    CC1=CC2=C(C=C1)N=C3[C@](C2=O)(CCN3C4=CC=CC=C4)O
1    1   716121           CN(C)CC[C@H](C1=CC=C(C=C1)Cl)C2=CC=CC=N2
2    2    65064  C1[C@H]([C@H](OC2=CC(=CC(=C21)O)O)C3=CC(=C(C(=...


In [25]:
from torch_geometric.data import Batch

# Build graph for every drug
print("Building drug graphs...")
drug_graphs = {}
failed = 0

for _, row in drugs_df.iterrows():
    cid = row["cid"]
    smi = row["smiles"]
    g = smiles_to_graph(str(smi))
    if g is not None:
        drug_graphs[cid] = g
    else:
        failed += 1

print(f"Graphs built: {len(drug_graphs)} / {len(drugs_df)}, failed: {failed}")

Building drug graphs...
Graphs built: 4130 / 4130, failed: 0


In [26]:
drug_enc.eval()
drug_embeddings = {}  # cid -> (64,) tensor

batch_size = 256
cid_list = list(drug_graphs.keys())

with torch.no_grad():
    for start in range(0, len(cid_list), batch_size):
        batch_cids = cid_list[start:start+batch_size]
        batch_graphs = [drug_graphs[c] for c in batch_cids]
        batch = Batch.from_data_list(batch_graphs).to(device)
        embs = drug_enc(batch.x, batch.edge_index, batch.batch)  # (B, 64)
        for cid, emb in zip(batch_cids, embs):
            drug_embeddings[cid] = emb.cpu()
        if start % 1000 == 0:
            print(f"  processed {start}/{len(cid_list)}")

print(f"Drug embeddings cached: {len(drug_embeddings)}")
print("Sample embedding shape:", next(iter(drug_embeddings.values())).shape)

  processed 0/4130
Drug embeddings cached: 4130
Sample embedding shape: torch.Size([64])


In [27]:
# pick first two drugs and test combination
sample_cids = list(drug_embeddings.keys())[:2]
d1 = drug_embeddings[sample_cids[0]].to(device)
d2 = drug_embeddings[sample_cids[1]].to(device)
print("Drug A embedding shape:", d1.shape)  # (64,)
print("Drug B embedding shape:", d2.shape)  # (64,)

# tumor embedding already confirmed as (1, 128)
# combined input to predictor will be: cat(tumor, drugA, drugB) = 128+64+64 = 256
print("Combined predictor input dim:", 128 + 64 + 64)

Drug A embedding shape: torch.Size([64])
Drug B embedding shape: torch.Size([64])
Combined predictor input dim: 256


In [51]:
class ResponsePredictor(nn.Module):
    def __init__(self, tumor_dim=128, drug_dim=64, hidden=256):
        super().__init__()
        in_dim = tumor_dim + drug_dim + drug_dim
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden, hidden // 2),
            nn.BatchNorm1d(hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, tumor_emb, drugA_emb, drugB_emb):
        x = torch.cat([tumor_emb, drugA_emb, drugB_emb], dim=-1)
        return self.net(x).squeeze(-1)


class MetaGNNComb(nn.Module):
    def __init__(self, n_genes=9253, tumor_dim=128, drug_dim=64):
        super().__init__()
        self.tumor_enc = TumorEncoder(n_genes=n_genes, tumor_dim=tumor_dim)
        self.drug_enc  = DrugEncoder(out_dim=drug_dim)
        self.predictor = ResponsePredictor(tumor_dim, drug_dim)

    def forward(self,
                expr_vec, coexpr_ei, ppi_ei, pathway_ei, ppi_ew, tumor_batch,
                drugA_batch, drugB_batch):
        tumor_emb = self.tumor_enc(
            expr_vec, coexpr_ei, ppi_ei, pathway_ei, ppi_ew, tumor_batch
        )  # (1, 128)

        dA_emb = self.drug_enc(
            drugA_batch.x, drugA_batch.edge_index, drugA_batch.batch
        )  # (B, 64)
        dB_emb = self.drug_enc(
            drugB_batch.x, drugB_batch.edge_index, drugB_batch.batch
        )  # (B, 64)

        # repeat tumor embedding to match drug batch size
        B = dA_emb.shape[0]
        tumor_emb = tumor_emb.expand(B, -1)  # (B, 128)

        return self.predictor(tumor_emb, dA_emb, dB_emb)  # (B,)


model = MetaGNNComb(n_genes=9253).to(device)
print(f"MetaGNNComb total params: {sum(p.numel() for p in model.parameters()):,}")

MetaGNNComb total params: 144,194


In [49]:
import pickle

# Load main label file
with open("cl_label_data_onlysmiles_1016.csv", "r") as f:
    label_data = pd.read_csv(f)

print("Type:", type(label_data))
if isinstance(label_data, pd.DataFrame):
    print("Shape:", label_data.shape)
    print("Columns:", label_data.columns.tolist())
    print(label_data.head(3))
elif isinstance(label_data, dict):
    print("Keys:", list(label_data.keys())[:5])
elif isinstance(label_data, list):
    print("Length:", len(label_data))
    print("First item:", label_data[0])

Type: <class 'pandas.DataFrame'>
Shape: (276087, 6)
Columns: ['drug_A', 'drug_B', 'cell_line_name', 'css_ri', 'S_sum', 'label']
   drug_A  drug_B  cell_line_name     css_ri      S_sum  label
0       8    3702             127  21.829123  25.515933      1
1      10    3702             127  28.148221  14.807248      1
2     331      11               9  44.665000  17.162000      1


In [46]:
# Load cell line counts
cl_counts = pd.read_csv("cl_label_data_onlysmiles_1016_cellline_counts_sorted.csv")
print("Columns:", cl_counts.columns.tolist())
print(cl_counts.head(5))
print("Shape:", cl_counts.shape)

Columns: ['细胞系', '对应样本量']
   细胞系  对应样本量
0  104     10
1    0     15
2   56     23
3   58     23
4   60     23
Shape: (140, 2)


In [33]:
# Rename columns
cl_counts.columns = ["cell_line_id", "sample_count"]
cl_counts = cl_counts.sort_values("sample_count", ascending=False).reset_index(drop=True)
print(cl_counts.head(10))
print("\nTotal cell lines:", len(cl_counts))

# MetaComb split: 127 data-rich, 13 data-poor
rich_cls = set(cl_counts.iloc[:127]["cell_line_id"].tolist())
poor_cls = set(cl_counts.iloc[127:]["cell_line_id"].tolist())
print("Rich cell lines:", len(rich_cls))
print("Poor cell lines:", len(poor_cls))

# Split label dataframe
rich_labels = label_data[label_data["cell_line_name"].isin(rich_cls)].reset_index(drop=True)
poor_labels = label_data[label_data["cell_line_name"].isin(poor_cls)].reset_index(drop=True)
print("Rich labels:", len(rich_labels))
print("Poor labels:", len(poor_labels))
print("Label balance (rich):", rich_labels["label"].value_counts().to_dict())

   cell_line_id  sample_count
0           131         10484
1           113          9965
2           130          9669
3            66          9613
4           115          9522
5           126          5794
6            52          5590
7            44          5575
8            92          5572
9             3          5563

Total cell lines: 140
Rich cell lines: 127
Poor cell lines: 13
Rich labels: 275678
Poor labels: 409
Label balance (rich): {0: 179278, 1: 96400}


In [34]:
import numpy as np

# Load dict_cell.npy — maps cell_line_id → row index in expression matrix
dict_cell = np.load("dict_cell.npy", allow_pickle=True).item()
print("Type:", type(dict_cell))
print("Num entries:", len(dict_cell))
print("Sample keys:", list(dict_cell.keys())[:5])
print("Sample values:", [dict_cell[k] for k in list(dict_cell.keys())[:5]])

Type: <class 'dict'>
Num entries: 140
Sample keys: ['22RV1_PROSTATE', '647V_URINARY_TRACT', 'A101D_SKIN', 'A2058_SKIN', 'A2780_OVARY']
Sample values: [0, 1, 2, 3, 4]


In [35]:
# Check what the integer IDs in label_data correspond to
# Load the cell line name list from expression matrix
expr_rich_df = pd.read_csv("log2_tpm_rich_zscore.csv", index_col=0)
cl_name_list = list(expr_rich_df.index)  # row order = dict_cell values
print("Total cell lines in expr:", len(cl_name_list))
print("First 5:", cl_name_list[:5])

# dict_cell maps name → row_idx
# We need: integer_id (from label_data) → row_idx in expression matrix
# Check if cl_counts cell_line_id matches row indices directly
print("\nSample cl_counts ids:", cl_counts["cell_line_id"].head(10).tolist())
print("Max cell_line_id:", cl_counts["cell_line_id"].max())
print("Min cell_line_id:", cl_counts["cell_line_id"].min())

# Check if label_data cell_line_name values are row indices
print("\nUnique cell_line_name values in label_data:", 
      sorted(label_data["cell_line_name"].unique())[:10])
print("Max:", label_data["cell_line_name"].max())

Total cell lines in expr: 127
First 5: [1, 2, 3, 4, 5]

Sample cl_counts ids: [131, 113, 130, 66, 115, 126, 52, 44, 92, 3]
Max cell_line_id: 139
Min cell_line_id: 0

Unique cell_line_name values in label_data: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Max: 139


In [36]:
# Load all 3 expression matrices (rich=127, poor=13, total=140)
expr_rich_df  = pd.read_csv("log2_tpm_rich_zscore.csv",          index_col=0)
expr_poor_df  = pd.read_csv("log2_tpm_poor_zscorebyrich.csv",    index_col=0)

# Rename gene columns
def rename_cols(df):
    df = df.rename(columns={c: c.replace(".", "-") for c in df.columns})
    return df.reindex(columns=shared_genes, fill_value=0.0)

expr_rich_df  = rename_cols(expr_rich_df)
expr_poor_df  = rename_cols(expr_poor_df)

print("Rich index sample:", expr_rich_df.index[:5].tolist())
print("Poor index sample:", expr_poor_df.index[:5].tolist())

# Combine into one dataframe
expr_all_df = pd.concat([expr_rich_df, expr_poor_df], axis=0)
print("Combined shape:", expr_all_df.shape)  # expect (140, 9253)
print("Combined index sample:", expr_all_df.index[:5].tolist())
print("Index dtype:", expr_all_df.index.dtype)

Rich index sample: [1, 2, 3, 4, 5]
Poor index sample: [0, 38, 41, 46, 56]
Combined shape: (140, 9253)
Combined index sample: [1, 2, 3, 4, 5]
Index dtype: int64


In [37]:
# cell_line_id (int) → expression numpy array (9253,)
cl_expr_lookup = {
    int(idx): expr_all_df.loc[idx].values.astype(np.float32)
    for idx in expr_all_df.index
}
print("Lookup size:", len(cl_expr_lookup))
print("Sample key:", list(cl_expr_lookup.keys())[:5])
print("Sample value shape:", cl_expr_lookup[1].shape)

# Verify all label_data cell lines are covered
label_cls = set(label_data["cell_line_name"].unique())
lookup_cls = set(cl_expr_lookup.keys())
missing = label_cls - lookup_cls
print("Missing cell lines:", missing)  # expect empty set

Lookup size: 140
Sample key: [1, 2, 3, 4, 5]
Sample value shape: (9253,)
Missing cell lines: set()


In [38]:
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Batch

class DrugComboDataset(Dataset):
    def __init__(self, label_df, cl_expr_lookup, drug_graphs):
        # filter out rows where drugA or drugB graph is missing
        mask = (
            label_df["drug_A"].isin(drug_graphs) &
            label_df["drug_B"].isin(drug_graphs) &
            label_df["cell_line_name"].isin(cl_expr_lookup)
        )
        self.df = label_df[mask].reset_index(drop=True)
        self.cl_expr_lookup = cl_expr_lookup
        self.drug_graphs    = drug_graphs
        print(f"Dataset size after filtering: {len(self.df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        cl_id  = int(row["cell_line_name"])
        dA_cid = int(row["drug_A"])
        dB_cid = int(row["drug_B"])
        label  = float(row["label"])

        expr   = torch.tensor(
            self.cl_expr_lookup[cl_id], dtype=torch.float32
        ).unsqueeze(1)  # (9253, 1)

        return {
            "expr":   expr,
            "graphA": self.drug_graphs[dA_cid],
            "graphB": self.drug_graphs[dB_cid],
            "label":  torch.tensor(label, dtype=torch.float32),
            "cl_id":  cl_id
        }


def collate_fn(batch):
    exprs  = torch.stack([b["expr"]  for b in batch], dim=0)   # (B, 9253, 1)
    labels = torch.stack([b["label"] for b in batch], dim=0)   # (B,)
    graphA = Batch.from_data_list([b["graphA"] for b in batch])
    graphB = Batch.from_data_list([b["graphB"] for b in batch])
    cl_ids = [b["cl_id"] for b in batch]
    return exprs, graphA, graphB, labels, cl_ids


# Build datasets
train_dataset = DrugComboDataset(rich_labels, cl_expr_lookup, drug_graphs)
test_dataset  = DrugComboDataset(poor_labels, cl_expr_lookup, drug_graphs)

train_loader  = DataLoader(
    train_dataset, batch_size=32, shuffle=True,
    collate_fn=collate_fn, num_workers=0
)
test_loader   = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    collate_fn=collate_fn, num_workers=0
)

print("Train batches:", len(train_loader))
print("Test batches: ", len(test_loader))

Dataset size after filtering: 275678
Dataset size after filtering: 409
Train batches: 8615
Test batches:  13


In [39]:
exprs, graphA, graphB, labels, cl_ids = next(iter(train_loader))
print("exprs shape:  ", exprs.shape)    # (32, 9253, 1)
print("labels shape: ", labels.shape)   # (32,)
print("graphA batch: ", graphA.batch.max().item() + 1)  # 32
print("graphB batch: ", graphB.batch.max().item() + 1)  # 32
print("label sample: ", labels[:5])

exprs shape:   torch.Size([32, 9253, 1])
labels shape:  torch.Size([32])
graphA batch:  32
graphB batch:  32
label sample:  tensor([0., 1., 1., 0., 0.])


In [40]:
pip install learn2learn

  Using cached learn2learn-0.2.0.tar.gz (7.0 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached gym-0.26.2-py3-none-any.whl
  Using cached torchvision-0.29.0-cp311-cp311-win_amd64.whl.metadata (5.7 kB)
  Using cached gsutil-5.37-py3-none-any.whl
  Using cached qpth-0.0.18-py3-none-any.whl
  Using cached gym_notices-0.1.0-py3-none-any.whl.metadata (1.2 kB)
  Using cached numpy-1.26.4-cp311-cp311-win_amd64.whl.metadata (61 kB)
  Using cached osqp-1.1.3-cp311-cp311-win_amd64.whl.metadata (2.2 kB)
  Using cached clarabel-0.11.1-cp39-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached scs-3.3.1

  error: subprocess-exited-with-error
  
  × Building wheel for learn2learn (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [154 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build\lib.win-amd64-cpython-311\learn2learn
      copying learn2learn\_version.py -> build\lib.win-amd64-cpython-311\learn2learn
      copying learn2learn\__init__.py -> build\lib.win-amd64-cpython-311\learn2learn
      creating build\lib.win-amd64-cpython-311\tests
      copying tests\__init__.py -> build\lib.win-amd64-cpython-311\tests
      creating build\lib.win-amd64-cpython-311\learn2learn\algorithms
      copying learn2learn\algorithms\base_learner.py -> build\lib.win-amd64-cpython-311\learn2learn\algorithms
      copying learn2learn\algorithms\gbml.py -> build\lib.win-amd64-cpython-311\learn2learn\algorithms
      copying learn2learn\algorithms\maml.py -> build\lib.win-amd64-cpython-311\learn2learn\algorithms
      copying learn2learn

In [41]:
import random
from collections import defaultdict

class CellLineTaskSampler:
    """
    Each task = one cell line.
    Returns K support samples + Q query samples from that cell line.
    """
    def __init__(self, label_df, cl_expr_lookup, drug_graphs,
                 n_support=10, n_query=10):
        self.n_support   = n_support
        self.n_query     = n_query
        self.cl_expr     = cl_expr_lookup
        self.drug_graphs = drug_graphs

        # group indices by cell line
        self.cl_to_indices = defaultdict(list)
        for idx, row in label_df.iterrows():
            cl  = int(row["cell_line_name"])
            dA  = int(row["drug_A"])
            dB  = int(row["drug_B"])
            if dA in drug_graphs and dB in drug_graphs and cl in cl_expr_lookup:
                self.cl_to_indices[cl].append(idx)

        # keep only cell lines with enough samples
        min_samples = n_support + n_query
        self.valid_cls = [
            cl for cl, idxs in self.cl_to_indices.items()
            if len(idxs) >= min_samples
        ]
        print(f"Valid cell lines for meta-training: {len(self.valid_cls)}")

    def sample_task(self, label_df):
        cl   = random.choice(self.valid_cls)
        idxs = self.cl_to_indices[cl]
        chosen = random.sample(idxs, self.n_support + self.n_query)
        support_idxs = chosen[:self.n_support]
        query_idxs   = chosen[self.n_support:]
        return cl, support_idxs, query_idxs

    def get_batch(self, label_df, idxs):
        rows   = label_df.loc[idxs]
        cl_id  = int(rows["cell_line_name"].iloc[0])
        expr   = torch.tensor(
            self.cl_expr[cl_id], dtype=torch.float32
        ).unsqueeze(1).to(device)                          # (9253, 1)
        batch  = torch.zeros(9253, dtype=torch.long).to(device)

        graphA = Batch.from_data_list(
            [self.drug_graphs[int(r["drug_A"])] for _, r in rows.iterrows()]
        ).to(device)
        graphB = Batch.from_data_list(
            [self.drug_graphs[int(r["drug_B"])] for _, r in rows.iterrows()]
        ).to(device)
        labels = torch.tensor(
            rows["label"].values, dtype=torch.float32
        ).to(device)

        return expr, batch, graphA, graphB, labels


task_sampler = CellLineTaskSampler(
    rich_labels, cl_expr_lookup, drug_graphs,
    n_support=10, n_query=10
)

Valid cell lines for meta-training: 127


In [52]:
import torch.optim as optim
from copy import deepcopy

# Hyperparameters
N_EPOCHS      = 50
N_TASKS       = 8        # tasks per meta-update
INNER_LR      =0.01
INNER_STEPS   = 3
OUTER_LR      = 1e-3
LOG_EVERY     = 5

# Fresh model
model     = MetaGNNComb(n_genes=9253).to(device)
meta_opt  = optim.Adam(model.parameters(), lr=OUTER_LR)
loss_fn   = nn.BCEWithLogitsLoss()

def inner_loop(fast_model, expr, batch, graphA, graphB, labels,
               inner_lr, inner_steps):
    """SGD adaptation on support set."""
    fast_model = deepcopy(fast_model)
    inner_opt  = optim.SGD(fast_model.parameters(), lr=inner_lr)
    for _ in range(inner_steps):
        logits = fast_model(
            expr, coexpr_ei, ppi_ei, pathway_ei, ppi_ew, batch,
            graphA, graphB
        )
        loss = loss_fn(logits, labels)
        inner_opt.zero_grad()
        loss.backward()
        inner_opt.step()
    return fast_model


print("Starting MAML training...")
for epoch in range(1, N_EPOCHS + 1):
    model.train()
    meta_opt.zero_grad()
    meta_loss = 0.0

    for _ in range(N_TASKS):
        # Sample task
        cl, sup_idxs, qry_idxs = task_sampler.sample_task(rich_labels)

        # Support set → inner loop adaptation
        s_expr, s_batch, s_gA, s_gB, s_labels = task_sampler.get_batch(
            rich_labels, sup_idxs
        )
        fast_model = inner_loop(
            model, s_expr, s_batch, s_gA, s_gB, s_labels,
            INNER_LR, INNER_STEPS
        )

        # Query set → meta loss
        q_expr, q_batch, q_gA, q_gB, q_labels = task_sampler.get_batch(
            rich_labels, qry_idxs
        )
        logits = fast_model(
            q_expr, coexpr_ei, ppi_ei, pathway_ei, ppi_ew, q_batch,
            q_gA, q_gB
        )
        task_loss = loss_fn(logits, q_labels)
        task_loss.backward()
        meta_loss += task_loss.item()

    # Outer update
    # clip gradients for stability
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    meta_opt.step()

    if epoch % LOG_EVERY == 0:
        print(f"Epoch {epoch:03d} | Meta Loss: {meta_loss/N_TASKS:.4f}")

print("Training complete.")

Starting MAML training...
Epoch 005 | Meta Loss: 0.7327
Epoch 010 | Meta Loss: 0.8196
Epoch 015 | Meta Loss: 0.7231
Epoch 020 | Meta Loss: 0.7194
Epoch 025 | Meta Loss: 0.7165
Epoch 030 | Meta Loss: 0.7122
Epoch 035 | Meta Loss: 0.7066
Epoch 040 | Meta Loss: 0.7298
Epoch 045 | Meta Loss: 0.7297
Epoch 050 | Meta Loss: 0.7132
Training complete.


In [53]:
torch.save({
    "model_state": model.state_dict(),
    "epoch": N_EPOCHS,
    "shared_genes": shared_genes,
}, "metagnn_comb_checkpoint.pt")
print("Checkpoint saved.")


Checkpoint saved.


In [54]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, confusion_matrix
)
import numpy as np
from copy import deepcopy

def evaluate_few_shot(model, label_df, cl_expr_lookup, drug_graphs,
                      coexpr_ei, ppi_ei, pathway_ei, ppi_ew,
                      n_support=10, inner_lr=0.01, inner_steps=3):
    """
    For each cell line in label_df:
      - sample n_support rows as support set → adapt
      - evaluate on remaining rows → collect predictions
    """
    model.eval()
    all_labels  = []
    all_probs   = []
    cl_results  = {}

    unique_cls = label_df["cell_line_name"].unique()

    for cl_id in unique_cls:
        cl_df = label_df[label_df["cell_line_name"] == cl_id].reset_index(drop=True)

        # filter valid drug graphs
        cl_df = cl_df[
            cl_df["drug_A"].isin(drug_graphs) &
            cl_df["drug_B"].isin(drug_graphs)
        ].reset_index(drop=True)

        if len(cl_df) < n_support + 1:
            print(f"  Skipping CL {cl_id}: only {len(cl_df)} samples")
            continue

        # Support / query split
        sup_df = cl_df.iloc[:n_support]
        qry_df = cl_df.iloc[n_support:]

        # Build support tensors
        expr  = torch.tensor(
            cl_expr_lookup[cl_id], dtype=torch.float32
        ).unsqueeze(1).to(device)
        batch = torch.zeros(expr.shape[0], dtype=torch.long).to(device)

        def make_batch(df):
            gA = Batch.from_data_list(
                [drug_graphs[int(r["drug_A"])] for _, r in df.iterrows()]
            ).to(device)
            gB = Batch.from_data_list(
                [drug_graphs[int(r["drug_B"])] for _, r in df.iterrows()]
            ).to(device)
            y  = torch.tensor(df["label"].values, dtype=torch.float32).to(device)
            return gA, gB, y

        # Inner loop adaptation on support set
        fast_model = deepcopy(model)
        inner_opt  = optim.SGD(fast_model.parameters(), lr=inner_lr)
        s_gA, s_gB, s_y = make_batch(sup_df)

        fast_model.train()
        for _ in range(inner_steps):
            logits = fast_model(
                expr, coexpr_ei, ppi_ei, pathway_ei, ppi_ew, batch,
                s_gA, s_gB
            )
            loss = loss_fn(logits, s_y)
            inner_opt.zero_grad()
            loss.backward()
            inner_opt.step()

        # Query evaluation
        fast_model.eval()
        q_gA, q_gB, q_y = make_batch(qry_df)
        with torch.no_grad():
            logits = fast_model(
                expr, coexpr_ei, ppi_ei, pathway_ei, ppi_ew, batch,
                q_gA, q_gB
            )
            probs = torch.sigmoid(logits).cpu().numpy()
            y_true = q_y.cpu().numpy()

        all_labels.extend(y_true)
        all_probs.extend(probs)

        # Per cell line AUROC
        if len(np.unique(y_true)) > 1:
            cl_auc = roc_auc_score(y_true, probs)
            cl_results[cl_id] = {"auc": cl_auc, "n": len(y_true)}
            print(f"  CL {cl_id:3d} | n={len(y_true):4d} | AUC={cl_auc:.4f}")

    return np.array(all_labels), np.array(all_probs), cl_results


print("=== Few-shot eval on DATA-POOR cell lines ===")
y_true, y_prob, cl_results = evaluate_few_shot(
    model, poor_labels, cl_expr_lookup, drug_graphs,
    coexpr_ei, ppi_ei, pathway_ei, ppi_ew,
    n_support=10, inner_lr=0.01, inner_steps=3
)

=== Few-shot eval on DATA-POOR cell lines ===
  CL 128 | n=  32 | AUC=0.5108
  CL  46 | n=  34 | AUC=0.3333
  CL  64 | n=  36 | AUC=0.6062
  CL  38 | n=  30 | AUC=0.6648
  CL  41 | n=  30 | AUC=0.5491
  CL  73 | n=  30 | AUC=0.4400
  CL 112 | n=  30 | AUC=0.4815
  CL   0 | n=   5 | AUC=0.5000
  CL  56 | n=  13 | AUC=0.9167
  CL  58 | n=  13 | AUC=0.7778
  CL  60 | n=  13 | AUC=0.7778
  CL 118 | n=  13 | AUC=0.5667
  Skipping CL 104: only 10 samples


In [58]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    accuracy_score, f1_score, confusion_matrix
)
import numpy as np
import json

y_pred = (y_prob >= 0.5).astype(int)

auroc  = roc_auc_score(y_true, y_prob)
auprc  = average_precision_score(y_true, y_prob)
acc    = accuracy_score(y_true, y_pred)
f1     = f1_score(y_true, y_pred)
cm     = confusion_matrix(y_true, y_pred)

print("=== Aggregate Results (Data-Poor Cell Lines) ===")
print(f"AUROC : {auroc:.4f}")
print(f"AUPRC : {auprc:.4f}")
print(f"Acc   : {acc:.4f}")
print(f"F1    : {f1:.4f}")
print(f"Confusion Matrix:\n{cm}")

aucs = [v["auc"] for v in cl_results.values()]
print(f"\n=== Per Cell Line AUC Summary ===")
print(f"Mean AUC : {np.mean(aucs):.4f}")
print(f"Std AUC  : {np.std(aucs):.4f}")
print(f"Min AUC  : {np.min(aucs):.4f}")
print(f"Max AUC  : {np.max(aucs):.4f}")

# Save
results_out = {
    "auroc": round(float(auroc), 4),
    "auprc": round(float(auprc), 4),
    "acc":   round(float(acc),   4),
    "f1":    round(float(f1),    4),
    "per_cl": {
        str(cl): {"auc": round(v["auc"], 4), "n": v["n"]}
        for cl, v in cl_results.items()
    }
}
with open("eval_results.json", "w") as f:
    json.dump(results_out, f, indent=2)
print("Saved eval_results.json")

=== Aggregate Results (Data-Poor Cell Lines) ===
AUROC : 0.6344
AUPRC : 0.6097
Acc   : 0.4229
F1    : 0.5924
Confusion Matrix:
[[  1 161]
 [  0 117]]

=== Per Cell Line AUC Summary ===
Mean AUC : 0.5937
Std AUC  : 0.1580
Min AUC  : 0.3333
Max AUC  : 0.9167
Saved eval_results.json
